# 🔥 ClusterOps: Thermal GPU Balancer — GRPO Training

**Train an LLM (Llama-3.1-8B) to manage a GPU data center under adversarial thermal conditions using Group Relative Policy Optimization (GRPO).**

---

## 1. Install Dependencies

In [ ]:
!pip install unsloth -q
!pip install trl>=0.12.0 transformers>=4.45.0 -q
!pip install fastapi uvicorn requests pydantic openenv-core -q
!pip install matplotlib numpy -q
print('✅ Dependencies installed')

## 2. Clone & Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os, sys

REPO_URL = "https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git"
REPO_DIR = "/content/clusterops_repo"

def setup_repo():
    if os.path.exists(REPO_DIR):
        print(f"Cleaning up existing directory: {REPO_DIR}")
        subprocess.run(["rm", "-rf", REPO_DIR])
    
    print(f"Cloning repository from {REPO_URL}...")
    result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ Git clone failed! Error: {result.stderr}")
        return False
    
    if not os.path.exists(REPO_DIR):
        print(f"❌ Directory {REPO_DIR} not found after clone!")
        return False
        
    os.chdir(REPO_DIR)
    print(f"✅ Successfully moved to {os.getcwd()}")
    return True

if setup_repo():
    # Start the FastAPI server in background
    print("Starting ClusterOps Environment Server...")
    server_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    # Wait for server to boot
    for i in range(15):
        time.sleep(2)
        try:
            resp = requests.get('http://localhost:8000/health', timeout=3)
            if resp.ok:
                print(f'✅ ClusterOps server online after {(i+1)*2}s')
                break
        except:
            print(f'  Waiting... ({(i+1)*2}s)')
    else:
        print('❌ Server failed to start. Logs:')
        print(server_proc.stderr.read().decode())

In [ ]:
# Verify server and show schema
import json
try:
    schema = requests.get('http://localhost:8000/schema').json()
    print('Environment Schema:')
    print(json.dumps(schema, indent=2))
except Exception as e:
    print(f"Could not fetch schema: {e}")

## 3. Baseline Agent (Before Training)

In [ ]:
ENV_URL = 'http://localhost:8000'

def env_reset(scenario='01_baseline'):
    return requests.post(f'{ENV_URL}/reset', json={'scenario': scenario}).json()

def env_step(action):
    return requests.post(f'{ENV_URL}/step', json=action).json()

def env_rubric():
    return requests.get(f'{ENV_URL}/grader/rubric').json()

def run_heuristic_baseline(scenario='01_baseline', verbose=False):
    data = env_reset(scenario)
    obs = data.get('observation', {})
    total_reward = 0.0

    while not data.get('done', False):
        nodes = obs.get('gpu_nodes', [])
        queue = obs.get('job_queue', [])
        action = {'action_type': 'wait'}
        for n in nodes:
            if n['status'] == 'busy' and n['temperature'] >= 90.0:
                action = {'action_type': 'evict', 'node_id': n['id']}
                break
        if action['action_type'] == 'wait' and queue:
            idle = sorted([n for n in nodes if n['status'] == 'idle'], key=lambda n: n['temperature'])
            if idle:
                action = {'action_type': 'allocate', 'job_id': queue[0]['id'], 'node_id': idle[0]['id']}

        data = env_step(action)
        obs = data.get('observation', {})
        total_reward += data.get('reward', 0)

    return total_reward, env_rubric()

print('Running 5 baseline episodes...')
baseline_rewards = []
for i in range(5):
    r, rub = run_heuristic_baseline('01_baseline')
    baseline_rewards.append(r)
print(f'📊 Baseline: avg_reward={sum(baseline_rewards)/5:.1f}')

## 4. Load LLM with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('✅ Model loaded with LoRA adapters')

## 5. GRPO Reward Functions

In [ ]:
import json, re

SYSTEM_PROMPT = """You are an autonomous GPU cluster scheduler. Your goal: schedule jobs to maximize completions and prevent thermal meltdowns.

Actions (respond with ONE JSON object only):
  {"action_type": "allocate", "job_id": "job_1", "node_id": 2}
  {"action_type": "evict", "node_id": 5}
  {"action_type": "cooldown", "node_id": 1}
  {"action_type": "wait"}

Prevent meltdowns (temp >= 100). Complete VIP jobs first."""

def parse_action(text):
    try:
        start, end = text.index('{'), text.rindex('}') + 1
        return json.loads(text[start:end])
    except:
        return {'action_type': 'wait'}

@torch.no_grad()
def run_llm_episode(model, tokenizer, scenario='01_baseline'):
    data = env_reset(scenario)
    total_reward = 0.0
    while not data.get('done', False):
        obs = data.get('observation', {})
        meta = data.get('metadata', {})
        prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\nStep {meta.get('step')}: {obs}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=60, temperature=0.5)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        data = env_step(parse_action(text))
        total_reward += data.get('reward', 0)
    return total_reward, env_rubric()

print('✅ GRPO logic ready')

## 6. Training & Results

In [ ]:
print("Starting training episodes...")
for ep in range(10):
    r, rub = run_llm_episode(model, tokenizer, '01_baseline')
    print(f"  Episode {ep+1}: Reward={r:.1f} Score={rub['total']:.3f}")

print("✅ Demo training complete. For full training, increase NUM_EPISODES.")